# 01: Load and clean the LendingClub extract

Loads the raw LendingClub "accepted loans" file, keeps resolved loans from 2015 onward, draws a 150,000-loan random sample, and writes `Dataset/loans_clean.csv`, which every later step uses.

**Before running**

1. Download `accepted_2007_to_2018Q4.csv` from Kaggle (`wordsforthewise/lending-club`); see `Documentation/dataset_guide.md`.
2. Place it at `Dataset/raw/accepted_2007_to_2018Q4.csv` (the raw file is git-ignored).
3. Run from the `Python/` folder; paths are relative to the repo root.

**What the cleaning does**

- Keeps only resolved loans (`Fully Paid`, `Charged Off`, `Default`). Loans still current or late have not finished playing out, so counting them as non-defaults would bias the model. This is a known trade-off: it distorts the recent vintage trend (see the README).
- Keeps vintages from 2015 onward, so the book is not dominated by small early cohorts.
- Draws a random sample of 150,000 loans (seed 42).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Run from the Python/ folder (or the repo root); paths are relative to the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == 'Python' else Path.cwd()
RAW_PATH = ROOT / 'Dataset' / 'raw' / 'accepted_2007_to_2018Q4.csv'
OUT_PATH = ROOT / 'Dataset' / 'loans_clean.csv'
MIN_VINTAGE_YEAR = 2015
SAMPLE_SIZE = 150_000   # set to None to keep everything after filtering
RANDOM_STATE = 42

KEEP_COLUMNS = [
    "id", "issue_d", "loan_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "emp_title", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "purpose", "addr_state", "dti",
    "delinq_2yrs", "fico_range_low", "fico_range_high", "inq_last_6mths",
    "open_acc", "revol_bal", "revol_util", "total_acc",
    "loan_status", "last_pymnt_d", "last_pymnt_amnt", "total_pymnt", "out_prncp",
]

RESOLVED_STATUSES = {"Fully Paid": 0, "Charged Off": 1, "Default": 1}

In [ ]:
def load_raw(path: Path) -> pd.DataFrame:
    # Read only the columns needed; the raw file is 151 columns wide.
    return pd.read_csv(path, usecols=lambda c: c in KEEP_COLUMNS, low_memory=False)


def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Resolve the target variable and drop unresolved loans
    df = df[df["loan_status"].isin(RESOLVED_STATUSES.keys())]
    df["defaulted"] = df["loan_status"].map(RESOLVED_STATUSES)

    # Parse issue_d into a date and a vintage month
    df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
    df = df.dropna(subset=["issue_d"])
    df["vintage_month"] = df["issue_d"].dt.to_period("M").astype(str)
    df = df[df["issue_d"].dt.year >= MIN_VINTAGE_YEAR]

    # Parse term ("36 months" -> 36)
    df["term_months"] = df["term"].astype(str).str.extract(r"(\d+)").astype(float)

    # Derive monthly income and the bureau score midpoint
    df["monthly_income"] = df["annual_inc"] / 12
    df["bureau_score"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

    # Duplicate and sanity filters
    df = df.drop_duplicates(subset=["id"])
    df = df[(df["loan_amnt"] > 0) & (df["monthly_income"] > 0)]

    # Report missingness before any filling (documented in Documentation/data_quality_notes.md)
    missing_summary = df.isna().mean().sort_values(ascending=False)
    missing_summary = missing_summary[missing_summary > 0]
    if len(missing_summary):
        print("Columns with missing values (share of the filtered loans, before filling and sampling):")
        print(missing_summary.round(4))

    # Fill a few columns where a simple fill is defensible; leave the rest as they are
    df["emp_length"] = df["emp_length"].fillna("Unknown")
    df["revol_util"] = df["revol_util"].fillna(df["revol_util"].median())
    df["dti"] = df["dti"].fillna(df["dti"].median())

    # Optional downsample to a tractable size
    if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
        df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

    return df.reset_index(drop=True)

In [1]:
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw file not found: {RAW_PATH}. See Documentation/dataset_guide.md for the download steps.")

print("Loading raw LendingClub file (this may take a minute; it is a large CSV)...")
raw = load_raw(RAW_PATH)
print(f"Raw rows loaded: {len(raw):,}")

cleaned = clean(raw)
print(f"\nRows in the final sample (resolved loans, {MIN_VINTAGE_YEAR}+ vintages): {len(cleaned):,}")
print(f"Default rate in cleaned set: {cleaned['defaulted'].mean():.2%}")

cleaned.to_csv(OUT_PATH, index=False)
print(f"\nCleaned dataset written to {OUT_PATH.relative_to(ROOT).as_posix()}")
print("Next: run Python/02_EDA_and_cleaning.ipynb.")

Loading raw LendingClub file (this may take a minute; it is a large CSV)...
Raw rows loaded: 2,260,701
Columns with missing values (share of the filtered loans, before filling and sampling):
emp_title         0.0654
emp_length        0.0645
last_pymnt_d      0.0022
revol_util        0.0006
dti               0.0000
inq_last_6mths    0.0000
dtype: float64

Rows in the final sample (resolved loans, 2015+ vintages): 150,000
Default rate in cleaned set: 21.55%

Cleaned dataset written to Dataset/loans_clean.csv
Next: run Python/02_EDA_and_cleaning.ipynb.
